# Durable Agent Orchestration

Durable orchestration lets a long-running agent resume safely after crashes, timeouts, approvals, or external delays.


## What to learn

- A checkpoint saves state after meaningful transitions.
- A stable run id reconnects later events to the correct task.
- Idempotency prevents a retried step from repeating a side effect.
- Compensation handles completed actions that cannot simply be rolled back.

## Decision rule

Persist state before and after side effects, give every action an idempotency key, separate pure reasoning from external writes, record approval events, and test recovery by interrupting runs at every boundary.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
# Import the dependencies used by this example.
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

# Define the data shape and small operations before running them.
class DeployState(TypedDict):
    artifact: str
    approved: bool
    status: str

def build(state: DeployState):
    return {"artifact": "app:v2", "status": "built"}

def approve(state: DeployState):
    decision = interrupt({"artifact": state["artifact"], "question": "Deploy it?"})
    return {"approved": bool(decision)}

def deploy(state: DeployState):
    # In production, pass state['artifact'] as an idempotency key.
    return {"status": "deployed" if state["approved"] else "cancelled"}

# Configure the framework object; this line prepares it but may not execute it yet.
builder = StateGraph(DeployState)
builder.add_node("build", build)
builder.add_node("approve", approve)
builder.add_node("deploy", deploy)
builder.add_edge(START, "build")
builder.add_edge("build", "approve")
builder.add_edge("approve", "deploy")
builder.add_edge("deploy", END)
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "deploy-42"}}
# Execute the configured model or workflow with the input below.
paused = graph.invoke({"artifact": "", "approved": False, "status": "new"}, config=config)
resumed = graph.invoke(Command(resume=True), config=config)
resumed


In [ ]:
"""Durable agent runner: checkpointing, crash recovery, and idempotent replay.
No API needed — the model, tools, and external service are simulated in pure
stdlib Python. We run a 4-step agent, crash the process after step 2's side
effect but BEFORE its checkpoint is written, then resume on the same thread:
completed steps are skipped, and the replayed step is deduplicated by its
idempotency key so the customer is charged exactly once."""
# Import the dependencies used by this example.
import json


# Define the data shape and small operations before running them.
class CrashError(RuntimeError):
    pass


class ExternalService:
    """Simulated payments/email API with server-side idempotency-key dedupe."""
    def __init__(self):
        self.executions = 0          # real side effects that actually happened
        self._processed = {}         # idempotency_key -> first result (24h-style window)

    def post(self, idem_key, action):
        if idem_key in self._processed:
            print(f"    [service] duplicate key {idem_key!r} -> cached result, effect suppressed")
            return self._processed[idem_key]
        self.executions += 1
        result = f"{action}#{self.executions:03d}"
        self._processed[idem_key] = result
        print(f"    [service] EXECUTED {action!r} (key={idem_key!r}) -> {result}")
        return result


class JsonCheckpointer:
    """Checkpoints as JSON blobs keyed by thread_id — a stand-in for
    SqliteSaver/PostgresSaver rows in LangGraph's BaseCheckpointSaver model."""
    def __init__(self):
        self._store = {}

    def save(self, thread_id, state):            # LangGraph: put()
        self._store[thread_id] = json.dumps(state)

    def load(self, thread_id):                   # LangGraph: get_tuple()
        raw = self._store.get(thread_id)
        return json.loads(raw) if raw else None


def make_steps(service):
    def plan(state, key):
        state["plan"] = ["fetch_order", "charge_customer", "send_receipt"]
        return state

    def fetch_order(state, key):                 # read-only: safe to replay
        state["order"] = {"id": "ord-42", "total_usd": 99}
        return state

    def charge_customer(state, key):             # write: MUST be idempotent
        state["charge_id"] = service.post(key, "charge $99")
        return state

    def send_receipt(state, key):                # write: MUST be idempotent
        state["receipt_id"] = service.post(key, "email receipt")
        return state

    return [("plan", plan), ("fetch_order", fetch_order),
            ("charge_customer", charge_customer), ("send_receipt", send_receipt)]


def run(thread_id, steps, saver, crash_after=None):
    """'sync' durability: checkpoint AFTER each step, BEFORE the next one."""
    state = saver.load(thread_id) or {"completed": 0}
    print(f"-- run(thread={thread_id!r}) starting at step {state['completed']}")
    for i, (name, fn) in enumerate(steps):
        if i < state["completed"]:
            print(f"  step {i} {name!r}: SKIP (already checkpointed)")
            continue
        idem_key = f"{thread_id}:{name}"         # deterministic across replays
        print(f"  step {i} {name!r}: RUN")
        state = fn(state, idem_key)              # side effect can happen here...
        if crash_after == i:                     # ...and the process can die here,
            raise CrashError(                    # after the effect, before the save
                f"process killed after step {i} ran, before its checkpoint")
        state["completed"] = i + 1
        saver.save(thread_id, state)             # durable save point
    print(f"-- run complete: charge_id={state['charge_id']}, receipt_id={state['receipt_id']}")
    return state


service, saver = ExternalService(), JsonCheckpointer()
steps = make_steps(service)

print("=== Attempt 1: crash mid-run (after the charge, before its checkpoint) ===")
try:
    run("thread-1", steps, saver, crash_after=2)
except CrashError as e:
    print(f"  !! CRASH: {e}")

print()
print("=== Attempt 2: resume the same thread from its checkpoint ===")
final = run("thread-1", steps, saver)

print()
print("=== Invariants ===")
print(f"external side effects executed: {service.executions} (expected 2: one charge, one receipt)")
assert service.executions == 2, "idempotency key failed to prevent a double charge"
assert final["completed"] == len(steps)
print("charge_customer replayed on resume, but the effect ran exactly once: OK")


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
